# Project Alpha: RL Bot v63p

## 📋 TODO & Roadmap
- **Data Policy:** Should we apply `handle_zeros_as_nan` globally to all matrices?
- **Feature Scaling:** Finalize the `get_agent_view` dual-view scaling logic.
- **Performance:** Move heavy logic (`analyze_4d_regime`) into `core` modular library.

<details>
<summary><b>📜 Version History (v59 - v63)</b></summary>

### v63
- Added **Metric Registry** (Strong-typed, dual-view).
- Fixed **Index Poisoning** bug (Dates in Ticker column).
- Unified **Perception** and **Execution** in Audit packs.
- Added 4 New Pillars: Return Autocorr, Range Position, OBV Divergence, Convexity.

### v62
- Fixed **Object Reference Mutation** bug in `compute_alpha_ensemble`.
- Added `PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`.

### v61
- Verified all metrics with Excel parity.
- Refactored `AlphaEngine` into Orchestrator pattern.

### v60
- Converted notebook code to modular system.
- Added Volatility Regime plots.

### v59
- Implemented **Result Pattern** for error handling.
- Switched to logarithmic returns globally.
</details>

## I. Initialization & Environment

In [1]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import os
import gc
import pandas as pd
import numpy as np
import multiprocessing
import random
import re
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional
from types import SimpleNamespace
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput, MarketObservation
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.quant import QuantUtils, TickerEngine
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_DIR = SU.load_env_from_root("DATA_DIR")
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"✓ Paths Initialized.\nDATA Dir: {DATA_DIR}\nOHLCV: {DATA_PATH_OHLCV}\nIndices: {DATA_PATH_INDICES}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

✓ Paths Initialized.
DATA Dir: c:\Users\ping\Files_win10\python\py311\stocks\data\
OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
Indices: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


## II. Data Pipeline

In [2]:
# === RELOADING FROM PARQUET ===
features_df = pd.read_parquet("output/features_df.parquet")
macro_df = pd.read_parquet("output/macro_df.parquet")
df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

In [3]:
print("⏳ Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

print("⚡ Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

⏳ Loading DataFrames... takes ~6 minutes
⚡ Generating Features...
⚡ Generating Decoupled Features (Benchmark: SPY)...
🚀 Unstacking Wide Matrices...
✅ Pipeline Complete. Tickers: 1578, Days: 16191


In [4]:
# === PERSISTENCE SANDBOX ===
features_df.to_parquet("output/features_df.parquet", index=True)
macro_df.to_parquet("output/macro_df.parquet", index=True)
df_close_wide.to_parquet("output/df_close_wide.parquet", index=True)
df_atrp_wide.to_parquet("output/df_atrp_wide.parquet", index=True)
df_trp_wide.to_parquet("output/df_trp_wide.parquet", index=True)

## III. The Analysis Suite

In [5]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

🎯 Master AlphaEngine Ready.


In [6]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=3,
    debug=False,
)

analyzer1, _ = create_walk_forward_analyzer(engine, _inputs, universe_subset=None)
analyzer1.show()

DEBUG: 941 stocks passed filters on 2026-04-16


In [7]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [8]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=3,
    debug=True,
)

print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df)
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
DEBUG: 941 stocks passed filters on 2026-04-16
----------------------------------------------------------------------
Timeline: [2026-04-01] -> Decision: 2026-04-16 -> Entry: 2026-04-17 -> End: 2026-04-24
Selected Tickers (3):
SOXL, CRDO, NBIS
----------------------------------------------------------------------


,Full,Lookback,Holding
Metric,,,
Group Gain,0.6818,0.5047,0.1557
Benchmark Gain,0.0858,0.0684,0.0053
== Gain Delta,0.5960,0.4363,0.1504
Group Sharpe,23.1077,24.1712,38.7616
Benchmark Sharpe,10.9495,14.0962,2.3401
== Sharpe Delta,12.1582,10.0750,36.4215
Group Sharpe (ATRP),0.6272,0.6934,0.5230
Benchmark Sharpe (ATRP),0.3892,0.4683,0.0879
== Sharpe (ATRP) Delta,0.2380,0.2251,0.4351



Target Reward Signal: 0.155737


In [9]:
##################################
##################################
##################################

In [10]:
result = SU.visualize_analyzer_structure(analyzer=analyzer1)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 3))
[  4]   📂 tickers (len=3)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]   📈 initial_weights (shape=(3,))
[  9]   📂 perf_metrics (len=24)
[ 10]     🔢 full_p_gain (float)
[ 11]     🔢 full_p_sharpe (float)
[ 12]     🔢 full_p_sharpe_atrp (float)
[ 13]     🔢 full_p_sharpe_trp (float)
[ 14]     🔢 lookback_p_gain (float)
[ 15]     🔢 lookback_p_sharpe (float)
[ 16]     🔢 lookback_p_sharpe_atrp (float)
[ 17]     🔢 lookback_p_sharpe_trp (float)
[ 18]     🔢 holding_p_gain (float)
[ 19]     🔢 holding_p_sharpe (float)
[ 20]     🔢 holding_p_sharpe_atrp (float)
[ 21]     🔢 holding_p_sharpe_trp (float)
[ 22]     🔢 full_b_gain (float)
[ 23]     🔢 full_b_sharpe (float)
[ 24]     🔢 full_b_sharpe_atrp (float)
[ 25]     🔢 full_b_sharpe_trp (float)
[ 26]     🔢 lookback_b_gain (flo

In [11]:
target_tickers = SU.peek(4, result)

 📍 INDEX: [4]
 🏷️  NAME:  tickers
 📂 PATH:  audit_pack -> tickers



['SOXL', 'CRDO', 'NBIS']

In [12]:
SU.export_audit_to_excel(
    audit_pack=analyzer1.last_run, filename="Audit_Verification_Report.xlsx"
)

📂 [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx
✨ Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR/output/Audit_Verification_Report.xlsx')

In [13]:
##################################
##################################
##################################

In [14]:
import pathlib

# Set display width to prevent line wrapping
pd.set_option("display.width", 2000)

# Show all columns instead of truncated view
pd.set_option("display.max_columns", None)


def load_latest_finviz_parquet(data_dir: str) -> pd.DataFrame:
    path = pathlib.Path(data_dir)
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"

    files = list(path.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {path.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Usage
df_finviz = load_latest_finviz_parquet(DATA_DIR)
print(f"df_finviz:\n{df_finviz}")

DEBUG: Loading file: 2026-05-01_df_finviz_merged_stocks_etfs.parquet
df_finviz:
       No.                                            Company               Index                  Sector                        Industry Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M    P/E  Fwd P/E   PEG    P/S    P/B    P/C  P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %    EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %   ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Pri

In [15]:
df_finviz.loc[target_tickers]

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
SOXL,143,Direxion Daily Semiconductor Bull 3X ETF,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, Equity - Leve...",17060.0,808,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.06,0.08,9/23/2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.14,11.88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.0215,1.62,149.52,111.04,164.07,977.69,210.25,5.55,8.51,6.5261,8.60,7.47,36.46,82.33,168.37,0.22,229.96,0.22,982.16,12.05 - 130.12,0.22,41809.39,75.00,-,3/11/2010,Yes,Yes,NaN,4.98,-2.18,NaN,87.25,0.46,40511820,NaN,126.98,124.21,131.38,121.66,130.40,2.69,Equity - Leveraged / Inverse,Equities (Stocks),0.75,43.0,17060.0,NaN,-32.97,NaN,-28.12,-12580.0,-42.44,976.07,115.1,26.07,"U.S., equity, semiconductors, leverage",23.3824,16093.1149,1434.6886,2.9973,4.7628,1.6001,7.9117,14.8420,3.2073,9.6206,18.2821,4.3910,7.3561,13.1625,3.1904,3.8448,6.1666,1.8611,2.7222,4.1252,1.5508,2.8591,4.4966,1.6220
CRDO,418,Credo Technology Group Holding Ltd,RUT,Technology,Semiconductors,Cayman Islands,NASD,"Technology, Semiconductors",34010.0,496,34010.0,102.26,NaN,0.31,31.84,18.37,26.13,119.88,10.04,7.06,NaN,NaN,-,0.0,1.80,1.02,NaN,NaN,NaN,NaN,NaN,201.49,408.64,6897.08,226.12,1070.0,339.76,17.38,0.21,184.22,165.63,89.91,10.20,-6.69,76.08,3.73,5.58,1.22,9.24,24.65,27.54,18.25,10.82,9.56,0.01,0.01,67.83,30.32,31.81,11.1258,-5.47,92.22,47.17,7.50,304.87,28.14,3.17,12.17,6.6005,7.70,7.83,17.54,44.46,34.68,-7.33,113.19,-13.76,308.37,45.15 - 213.80,-13.76,2462.61,64.87,Mar 02/a,1/27/2022,Yes,Yes,622.0,4.76,1.14,1.16,7.55,0.62,4661626,212.16,174.01,176.00,184.88,171.48,184.38,5.96,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,7.9443,54.3643,5.8432,1.6289,2.6794,1.3022,2.0810,3.1288,1.3563,5.5665,11.7919,2.4392,5.4367,10.8443,2.3447,3.1077,5.3161,1.6318,0.6828,1.0507,1.1175,1.9934,3.1857,1.3862
NBIS,376,Nebius Group NV,-,Technology,Software - Infrastructure,Netherlands,NASD,"Technology, Software - Infrastructure",38890.0,445,38890.0,303.10,NaN,NaN,72.79,8.47,10.57,NaN,18.23,14.61,NaN,NaN,-,0.0,0.51,-0.77,NaN,NaN,NaN,NaN,NaN,500.79,-70.26,373.88,354.32,534.2,101.60,5.74,-7.92,219.47,201.25,91.70,20.05,-0.26,39.84,1.04,21.68,2.71,43.63,0.25,0.50,1.07,3.08,6.57,1.05,1.06,-7.66,-113.33,19.02,14.0063,4.98,51.54,81.35,23.49,545.73,84.56,3.87,11.37,7.3597,8.27,8.04,5.37,27.00,56.31,-8.43,84.95,-8.43,564.47,23.25 - 168.71,-8.43,1454.21,60.25,May 13/b,5/24/2011,Yes,Yes,NaN,9.27,2.29,1.71,16.07,0.98,15709917,176.85,138.23,141.39,156.00,140.00,154.49,11.76,,,NaN,NaN,NaN,NaN,NaN,

In [16]:
print(df_finviz.loc[target_tickers])

      No.                                   Company Index      Sector                   Industry         Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M     P/E  Fwd P/E   PEG    P/S    P/B    P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %   EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %  ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %  52W Low %   

In [17]:
def cleanup_ui():
    clear_output(wait=True)
    pio.renderers.default = None
    gc.collect()
    print("🧹 Memory Cleared.")


def active_engine_audit():
    gc.collect()
    engines = [obj for obj in gc.get_objects() if type(obj).__name__ == "AlphaEngine"]
    print(f"Active AlphaEngine Instances: {len(engines)}")


active_engine_audit()

Active AlphaEngine Instances: 1


In [18]:
# print(features_df.info())
# display(features_df.head())
# display(df_indices.tail())

In [19]:
################################
################################
################################
################################

In [20]:
SU.peek(0, result)

 📍 INDEX: [0]
 🏷️  NAME:  audit_pack
 📂 PATH:  audit_pack



EngineOutput(portfolio_series=Date
2026-04-01    1.0000
2026-04-02    1.0448
2026-04-06    1.0736
2026-04-07    1.1157
2026-04-08    1.2222
2026-04-09    1.2799
2026-04-10    1.3768
2026-04-13    1.4861
2026-04-14    1.6280
2026-04-15    1.6786
2026-04-16    1.6565
2026-04-17    1.6761
2026-04-20    1.7388
2026-04-21    1.7746
2026-04-22    1.8428
2026-04-23    1.8776
2026-04-24    1.9774
dtype: float64, benchmark_series=Date
2026-04-01    1.0000
2026-04-02    1.0009
2026-04-06    1.0056
2026-04-07    1.0061
2026-04-08    1.0317
2026-04-09    1.0377
2026-04-10    1.0370
2026-04-13    1.0471
2026-04-14    1.0599
2026-04-15    1.0682
2026-04-16    1.0708
2026-04-17    1.0838
2026-04-20    1.0816
2026-04-21    1.0745
2026-04-22    1.0854
2026-04-23    1.0812
2026-04-24    1.0896
dtype: float64, normalized_plot_data=Ticker        CRDO    SOXL    NBIS
Date                              
2026-04-01  1.0000  1.0000  1.0000
2026-04-02  1.0577  1.0094  1.0674
2026-04-06  1.0682  1.0488  1.1039
2

EngineOutput(portfolio_series=Date
2026-04-01    1.0000
2026-04-02    1.0448
2026-04-06    1.0736
2026-04-07    1.1157
2026-04-08    1.2222
2026-04-09    1.2799
2026-04-10    1.3768
2026-04-13    1.4861
2026-04-14    1.6280
2026-04-15    1.6786
2026-04-16    1.6565
2026-04-17    1.6761
2026-04-20    1.7388
2026-04-21    1.7746
2026-04-22    1.8428
2026-04-23    1.8776
2026-04-24    1.9774
dtype: float64, benchmark_series=Date
2026-04-01    1.0000
2026-04-02    1.0009
2026-04-06    1.0056
2026-04-07    1.0061
2026-04-08    1.0317
2026-04-09    1.0377
2026-04-10    1.0370
2026-04-13    1.0471
2026-04-14    1.0599
2026-04-15    1.0682
2026-04-16    1.0708
2026-04-17    1.0838
2026-04-20    1.0816
2026-04-21    1.0745
2026-04-22    1.0854
2026-04-23    1.0812
2026-04-24    1.0896
dtype: float64, normalized_plot_data=Ticker        CRDO    SOXL    NBIS
Date                              
2026-04-01  1.0000  1.0000  1.0000
2026-04-02  1.0577  1.0094  1.0674
2026-04-06  1.0682  1.0488  1.1039
2

In [21]:
features_df.loc["TSLA"]

,ATR,ATRP,TRP,RSI,Mom_21,Consistency,IR_63,Beta_63,DD_21,AutoCorr_15,Ret_1d,Range_Pos_20,Slope_P_5,Slope_V_5,Convexity,RollingStalePct,RollMedDollarVol,RollingSameVolCount
Date,,,,,,,,,,,,,,,,,,
2010-06-29,NaN,0.0000,0.0000,50.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-06-30,0.4747,0.2988,0.2988,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,-0.0025,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-01,0.4677,0.3194,0.2573,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,-0.0785,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-02,0.4552,0.3556,0.2286,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000,-0.1257,NaN,NaN,NaN,0.0000,NaN,NaN,NaN
2010-07-06,0.4425,0.4120,0.2588,0.0000,NaN,0.0,NaN,1.0000,0.0000,0.0000,-0.1609,NaN,-0.1004,-0.6060,0.0000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-27,15.0660,0.0398,0.0443,50.0475,0.0176,0.6,-0.1271,1.6289,-0.0548,0.1725,0.0063,0.5751,-0.0070,0.2600,0.0082,0.0,2.8928e+10,0.0
2026-04-28,14.6863,0.0391,0.0259,48.7541,0.0392,0.6,-0.1180,1.6338,-0.0614,0.1197,-0.0070,0.5383,-0.0047,0.1559,0.0071,0.0,2.8875e+10,0.0
2026-04-29,14.0916,0.0378,0.0171,47.1592,0.0493,0.4,-0.1258,1.6349,-0.0694,0.1039,-0.0086,0.4936,-0.0006,0.1315,0.0064,0.0,2.8776e+10,0.0
